# 01 — Triplet Parsing (CholecT50)

**Goal:** Load the CholecT50 JSON annotations, understand the binary-to-label mapping,
and parse the surgical action triplets `(instrument, verb, target)` for 3 sample videos.

**Videos:** VID01 (short), VID05 (medium), VID40 (longer)

In [ ]:
import json
import os
import pandas as pd
import numpy as np
from pathlib import Path

# Project paths
PROJECT_ROOT = Path('..').resolve()
DATA_DIR = PROJECT_ROOT / 'data' / 'CholecT45'
TRIPLET_DIR = DATA_DIR / 'triplet'
DICT_DIR = DATA_DIR / 'dict'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'parsed_triplets'

print('Project root:', PROJECT_ROOT)
print('Data dir:', DATA_DIR)
print('Available label files:', os.listdir(TRIPLET_DIR))
print('Dict contents:', os.listdir(DICT_DIR))

## 1. Load and Inspect a Single Video's JSON

Each JSON file has the following top-level keys:
- `categories` — dictionaries mapping IDs to label names (instrument, verb, target, triplet)
- `annotations` — per-frame annotation arrays
- `num_frames`, `fps`, `video`, `info`, `licenses`

In [ ]:
# Load VID01 as our test case
with open(TRIPLET_DIR / 'VID01.json') as f:
    vid01 = json.load(f)

print('Top-level keys:', list(vid01.keys()))
print(f"Video: {vid01['video']}")
print(f"FPS: {vid01['fps']}")
print(f"Num frames: {vid01['num_frames']}")
print(f"Annotated frames: {len(vid01['annotations'])}")

## 2. Inspect the Category Dictionaries

These are our binary-to-label mappings — the key piece for decoding the annotations.

In [ ]:
categories = vid01['categories']

instrument_map = categories['instrument']
verb_map = categories['verb']
target_map = categories['target']
triplet_map = categories['triplet']

print('=== INSTRUMENTS (6 classes) ===')
for k, v in instrument_map.items():
    print(f'  {k}: {v}')

print('\n=== VERBS (10 classes) ===')
for k, v in verb_map.items():
    print(f'  {k}: {v}')

print('\n=== TARGETS (15 classes) ===')
for k, v in target_map.items():
    print(f'  {k}: {v}')

print(f'\n=== TRIPLETS ({len(triplet_map)} classes) ===')
for k, v in list(triplet_map.items())[:10]:
    print(f'  {k}: {v}')
print(f'  ... ({len(triplet_map)} total)')

## 3. Understand the Annotation Format

Each frame annotation is a list of entries. Each entry is a **15-element array**:

```
[triplet_id, instrument_id, presence,
 bbox_x, bbox_y, bbox_w, bbox_h,
 verb_id, target_id, presence,
 bbox_x, bbox_y, bbox_w, bbox_h,
 phase_annotation]
```

Multi-label frames have **multiple entries** in the list (multiple active triplets).

In [ ]:
annotations = vid01['annotations']

# Inspect first few frames
for frame_id in ['0', '1', '2']:
    print(f'Frame {frame_id}: {annotations[frame_id]}')
    print(f'  -> {len(annotations[frame_id])} triplet(s) active')
    print()

# Find a multi-label frame
multi_label_frames = {k: v for k, v in annotations.items() if len(v) > 1}
print(f'Frames with multiple active triplets: {len(multi_label_frames)} / {len(annotations)}')
print(f'Ratio: {len(multi_label_frames)/len(annotations)*100:.1f}%')

# Show one multi-label frame
if multi_label_frames:
    sample_key = list(multi_label_frames.keys())[0]
    print(f'\nExample multi-label frame {sample_key}:')
    for i, entry in enumerate(multi_label_frames[sample_key]):
        triplet_id = entry[0]
        triplet_label = triplet_map.get(str(triplet_id), 'UNKNOWN')
        print(f'  Entry {i}: triplet_id={triplet_id} -> {triplet_label}')

## 4. Decode Annotations Into Readable Triplets

We extract `(instrument, verb, target)` from each frame's annotation entries.

In [ ]:
def decode_annotation_entry(entry, categories):
    """Decode a single 15-element annotation entry into a readable triplet."""
    triplet_id = entry[0]
    instrument_id = entry[1]
    verb_id = entry[7]
    target_id = entry[8]
    
    instrument = categories['instrument'].get(str(instrument_id), f'unknown_{instrument_id}')
    verb = categories['verb'].get(str(verb_id), f'unknown_{verb_id}')
    target = categories['target'].get(str(target_id), f'unknown_{target_id}')
    triplet_label = categories['triplet'].get(str(triplet_id), f'unknown_{triplet_id}')
    
    return {
        'triplet_id': triplet_id,
        'triplet_label': triplet_label,
        'instrument_id': instrument_id,
        'instrument': instrument,
        'verb_id': verb_id,
        'verb': verb,
        'target_id': target_id,
        'target': target
    }

# Test on frame 0
test_entry = annotations['0'][0]
decoded = decode_annotation_entry(test_entry, categories)
print('Frame 0, Entry 0:')
for k, v in decoded.items():
    print(f'  {k}: {v}')

In [ ]:
def parse_video_annotations(json_path):
    """Parse all frames of a video JSON into a DataFrame of decoded triplets.
    
    Handles multi-label frames: each active triplet becomes its own row.
    """
    with open(json_path) as f:
        data = json.load(f)
    
    categories = data['categories']
    annotations = data['annotations']
    video_name = data.get('video', os.path.basename(json_path))
    
    rows = []
    for frame_id, entries in annotations.items():
        for entry in entries:
            decoded = decode_annotation_entry(entry, categories)
            decoded['frame'] = int(frame_id)
            decoded['video'] = video_name
            decoded['num_triplets_in_frame'] = len(entries)
            rows.append(decoded)
    
    df = pd.DataFrame(rows)
    # Sort by frame number
    df = df.sort_values('frame').reset_index(drop=True)
    
    # Reorder columns for readability
    col_order = ['video', 'frame', 'num_triplets_in_frame',
                 'triplet_id', 'triplet_label',
                 'instrument_id', 'instrument',
                 'verb_id', 'verb',
                 'target_id', 'target']
    df = df[col_order]
    return df

# Parse VID01
df_vid01 = parse_video_annotations(TRIPLET_DIR / 'VID01.json')
print(f'VID01: {len(df_vid01)} total triplet annotations across {df_vid01["frame"].nunique()} frames')
df_vid01.head(10)

## 5. Multi-Label Frame Analysis

How often do frames have more than one active triplet?

In [ ]:
# Distribution of triplets per frame
triplets_per_frame = df_vid01.groupby('frame').size()

print('=== Triplets per Frame Distribution (VID01) ===')
print(triplets_per_frame.value_counts().sort_index())
print(f'\nMax triplets in a single frame: {triplets_per_frame.max()}')
print(f'Mean triplets per frame: {triplets_per_frame.mean():.2f}')
print(f'Frames with >1 triplet: {(triplets_per_frame > 1).sum()} / {len(triplets_per_frame)}')

In [ ]:
# Show some multi-label frame examples
multi_frames = df_vid01[df_vid01['num_triplets_in_frame'] > 1]
if not multi_frames.empty:
    sample_frame = multi_frames['frame'].iloc[0]
    print(f'=== Multi-label Frame {sample_frame} ===')
    display(df_vid01[df_vid01['frame'] == sample_frame][['frame', 'instrument', 'verb', 'target', 'triplet_label']])

## 6. Summary Statistics

In [ ]:
print('=== VID01 Label Frequency ===')
print('\nTop 10 Triplets:')
print(df_vid01['triplet_label'].value_counts().head(10))
print('\nInstruments:')
print(df_vid01['instrument'].value_counts())
print('\nVerbs:')
print(df_vid01['verb'].value_counts())
print('\nTargets:')
print(df_vid01['target'].value_counts())

---
**Next steps:** In the next cells we will extend parsing to all 3 videos and save CSVs.